# 数据加载：memmap + ShuffledBlockSampler

源码导航：[`train/walkie_pretrain.py`](../../../train/walkie_pretrain.py) 中的 `_batch_from_starts`、`get_batch`、`ShuffledBlockSampler`。

大规模预训练语料（通常数十 GB 到数 TB 的 Token 序列）无法完整加载到内存。Walkie 使用 `numpy.memmap` 将 token 数组直接映射到磁盘文件，操作系统的页缓存机制按需将活跃的块调入内存，无需手动管理 IO 缓冲区。

在此基础上，`ShuffledBlockSampler` 实现了**无放回 shuffle 的 block 级采样**：将整个 token 序列划分为固定长度的 block，每轮（epoch）对 block 索引进行 shuffle，然后按顺序依次发出 batch，避免同一 batch 内重复看到相同的 token 片段。

### 1. 理论背景

设 token 总数为 $N$，block_size 为 $L$，则可用样本数为 $n = \lfloor N / L \rfloor$（末尾不足一个 block 的 token 被舍弃）。每个样本 $i$ 对应起始偏移 $s_i = \text{order}[i] \times L$，对应的输入序列为 $x = \text{data}[s_i : s_i + L]$，标签序列为 $y = \text{data}[s_i + 1 : s_i + L + 1]$（next-token prediction）。

**Epoch 与 Cursor 机制**：

```
order = shuffle(arange(n))   # 对 n 个 block 索引洗牌
batch_t: starts = order[cursor : cursor + batch_size] * L
cursor += global_batch_size
if cursor + global_batch_size > n:
    epoch += 1; cursor = 0; reshuffle(seed + epoch)
```

每个 epoch 使用确定性种子 `seed + epoch` 重新洗牌，保证训练可复现，且 checkpoint 只需保存 `(epoch, cursor)` 即可完整恢复数据位置。

**DDP 分片**：在 `world_size` 个 GPU 的分布式训练中，Rank $r$ 取全局 batch 中的第 $r$ 个 `batch_size` 切片：

$$
\text{starts}_r = \text{order}[\text{cursor} + r \times B : \text{cursor} + (r+1) \times B] \times L
$$

各 rank 各自独立读取磁盘，无需跨进程通信。

### 2. memmap 的内存模型

`np.memmap(path, dtype=np.uint16, mode='r')` 创建一个只读文件映射，返回的数组在访问时触发操作系统的**按需缺页加载（demand paging）**。token 文件以 `uint16` 存储（词表 $\leq 65535$）时，1GB 文件对应约 500M tokens。训练过程中热点块会被 OS 缓存在页缓存中，冷块则在需要时从 SSD/HDD 加载。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 直接从训练脚本导入相关工具函数
from train.walkie_pretrain import ShuffledBlockSampler, get_batch, _batch_from_starts

### 3. 用随机内存数组模拟 memmap 数据

In [ ]:
VOCAB_SIZE  = 128
BLOCK_SIZE  = 16
BATCH_SIZE  = 4

# 模拟一段 token 序列（实际为 np.memmap）
rng = np.random.default_rng(42)
fake_data = rng.integers(0, VOCAB_SIZE, size=10_000, dtype=np.int64)

# get_batch：随机采样（用于 eval 或简单基线）
x, y = get_batch(fake_data, BLOCK_SIZE, BATCH_SIZE, device=torch.device('cpu'))
print("x shape:", tuple(x.shape), " y shape:", tuple(y.shape))

# 验证 x 与 y 的 next-token 关系：y[i] 应为 x[i] 往后移一位
# 即对同一起始位置 s：x = data[s:s+L], y = data[s+1:s+L+1]
# 我们检查 x 的最后一个 token + 1 的位置是否等于 y 的最后一个 token
print("x[0,-3:] =", x[0, -3:].tolist())
print("y[0,-3:] =", y[0, -3:].tolist(), " (x 右移一位)")

### 4. ShuffledBlockSampler：无放回 shuffle 验证

In [ ]:
sampler = ShuffledBlockSampler(
    data=fake_data,
    block_size=BLOCK_SIZE,
    batch_size=BATCH_SIZE,
    device=torch.device('cpu'),
    seed=0,
    rank=0,
    world_size=1,
    name='train',
)

print(f"num_samples: {sampler.num_samples}  (≈ {len(fake_data)} / {BLOCK_SIZE})")
print(f"tokens_per_epoch: {sampler.tokens_per_epoch:,}")

# 采集一个 epoch 内的所有 block 起始索引，验证无放回
steps_per_epoch = sampler.num_samples // sampler.global_batch_size
all_starts = []
for _ in range(steps_per_epoch):
    starts = sampler.next_starts()
    all_starts.extend(starts.tolist())

print(f"\n采集 step 数: {steps_per_epoch}，总 block 数: {len(all_starts)}")
print(f"唯一 start 数: {len(set(all_starts))}  (无放回 → 应与总 block 数相等)")

# 验证 state_dict 保存与恢复
state = sampler.state_dict()
print(f"\nstate_dict keys: {list(state.keys())}")
print(f"epoch={state['epoch']}, cursor={state['cursor']}")

### 5. 源码精讲

**`_batch_from_starts`**（将 block 起始位置转为 Tensor batch）：

```python
def _batch_from_starts(data, starts, block_size, device):
    # starts: (batch_size,) 的起始偏移数组
    # 广播构造 (batch_size, block_size) 的索引矩阵
    offsets = starts[:, None] + np.arange(block_size)[None, :]   # (B, L)
    x = torch.from_numpy(np.asarray(data[offsets], dtype=np.int64))
    y = torch.from_numpy(np.asarray(data[offsets + 1], dtype=np.int64))  # next-token
    return x.to(device), y.to(device)
```

**`ShuffledBlockSampler.next_starts()`**（无放回顺序发出一批 block 起始位置）：

```python
def next_starts(self):
    self._ensure_room_for_global_batch()  # 不够时触发 epoch++, reshuffle
    # rank r 取第 r 个 batch_size 切片
    begin = self.cursor + self.rank * self.batch_size
    end   = begin + self.batch_size
    starts = self.order[begin:end].astype(np.int64) * self.block_size
    self.cursor += self.global_batch_size
    return starts

def _reshuffle(self):
    self.order[:] = np.arange(self.num_samples)
    # 确定性种子：seed + epoch，保证 checkpoint 可复现
    rng = np.random.default_rng(self.seed + self.epoch)
    rng.shuffle(self.order)
```

---

## 延伸阅读与参考资料

### NumPy / OS
- **numpy.memmap**: [docs](https://numpy.org/doc/stable/reference/generated/numpy.memmap.html)
- **Linux 虚拟内存与页缓存**: Understanding the Linux Kernel, Bovet & Cesati

### 大规模数据流水线
- **The Pile 数据格式（jsonl + token bin）**: Gao et al., 2020. [arXiv:2101.00027](https://arxiv.org/abs/2101.00027)
- **nanoGPT 数据准备脚本**: [GitHub](https://github.com/karpathy/nanoGPT/blob/master/data/openwebtext/prepare.py)